In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import QuantileTransformer, OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, train_test_split
from xgboost import XGBClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier
from sklearn.svm import SVC


In [ ]:
path = r"..\data\Students Social Media Addiction.csv"
data = pd.read_csv(path)

X = data.drop(['Student_ID','Addicted_Score'],axis = 1)
Y = data['Addicted_Score']

x_train ,x_test, y_train, y_test = train_test_split(X,Y, test_size=0.25,random_state=42)

In [ ]:
column_onehot = ['Relationship_Status','Academic_Level','Country','Most_Used_Platform','Gender', 'Affects_Academic_Performance']
onehoten = Pipeline(steps=[
  ('onehot',OneHotEncoder(handle_unknown = 'ignore',sparse_output=False))
  ])

In [ ]:
preprocess = ColumnTransformer(transformers=[
  ("onehot",onehoten,column_onehot)
],remainder='passthrough'
)

In [ ]:
stackmodel = StackingClassifier(estimators=[
  ("model_knn",KNeighborsClassifier(n_neighbors=6)),
  ("model_xg",XGBClassifier()),
], final_estimator=SVC(),cv=4,n_jobs=-1)

In [ ]:
steps = [
  ('pre',preprocess),
  ('scale',QuantileTransformer(n_quantiles=100)),
  ('estimator',stackmodel)
]

pipe = Pipeline(steps)
pipe.fit(x_train,y_train)

In [ ]:
scores = cross_val_score(pipe,x_train,y_train,cv=5,scoring='accuracy',n_jobs=-1)

print("CV scores:", scores)
print("Mean accuracy:", scores.mean())
print("Std deviation:", scores.std())